<div style='text-align: center; padding: 30px; background: linear-gradient(135deg, #0f0c29 0%, #302b63 50%, #24243e 100%); border-radius: 15px; margin: 10px 0; box-shadow: 0 10px 30px rgba(0,0,0,0.3);'>
  <h1 style='color: #ffffff; margin: 0 0 8px 0; font-size: 2.5em;'>🎬 LTX-Video Generator</h1>
  <h3 style='color: #c0c0ff; margin: 0 0 5px 0; font-weight: 400;'>Kaggle T4 GPU Edition</h3>
  <p style='color: #aaa; margin: 0; text-align: center;'>Text-to-Video • Image-to-Video | HuggingFace Diffusers + Gradio</p>
</div>

---

### 🚀 What is this notebook?

High-quality **Text-to-Video** and **Image-to-Video** generation using the official **Lightricks LTX-Video** model.

| Feature | Detail |
|---|---|
| **Model** | Lightricks/LTX-Video (9B parameters) |
| **GPU** | T4 x1 (16 GB VRAM) |
| **Engine** | HuggingFace Diffusers |
| **Memory** | Sequential CPU Offloading |
| **Modes** | Text-to-Video, Image-to-Video |
| **Extras** | Audio support, Video Gallery, Advanced parameters |

### ⚡ Quick Start
1. **Settings → Accelerator → GPU T4 x1**
2. **Turn on Internet** in Settings sidebar
3. Run all cells in order (Runtime → Run All)
4. Open the Gradio URL and start generating!

---
## Step 1: Environment Setup

Optimizes memory and configures PyTorch for Kaggle T4 GPU.

In [ ]:
import os, gc, psutil

print('=' * 50)
print('  Kaggle T4 Environment Setup')
print('=' * 50)

mem = psutil.virtual_memory()
print(f'  RAM  : {mem.total / 1024**3:.1f} GB total, {mem.available / 1024**3:.1f} GB available')

os.system('echo 3 | sudo tee /proc/sys/vm/drop_caches > /dev/null 2>&1')
os.system('echo 1 | sudo tee /proc/sys/vm/overcommit_memory > /dev/null 2>&1')
gc.collect()

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,garbage_collection_threshold:0.6'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['OMP_NUM_THREADS'] = '4'

print('  ✓ Environment optimized for T4 GPU')
print('=' * 50)

---
## Step 2: Install Dependencies

Installs HuggingFace Diffusers, Gradio, and supporting libraries.

In [ ]:
import subprocess

try:
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                           capture_output=True, text=True, check=True)
    print(f'  GPU: {result.stdout.strip()}')
except Exception:
    print('  ⚠ WARNING: No GPU detected! Settings → Accelerator → GPU T4 x1')

!pip install -q --no-deps torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1 --index-url https://download.pytorch.org/whl/cu121

print('  ✓ PyTorch 2.3.1 installed')
print('  ⚠ IMPORTANT: After this cell finishes, click Runtime → Restart & Run All')

!pip install -q diffusers transformers accelerate gradio imageio[ffmpeg] soundfile huggingface_hub

---
## Step 3: Model Cache Setup

Downloads the LTX-Video model from HuggingFace with smart caching.
First run downloads ~18 GB. Subsequent runs use cache.

In [ ]:
import os
from huggingface_hub import snapshot_download

MODEL_ID = 'Lightricks/LTX-Video'
CACHE_DIR = '/kaggle/working/model_cache'

print(f'  Model: {MODEL_ID}')
print(f'  Cache: {CACHE_DIR}')

if not os.path.exists(os.path.join(CACHE_DIR, 'model_index.json')):
    print('  Downloading model (one-time, ~18 GB)...')
    snapshot_download(repo_id=MODEL_ID, local_dir=CACHE_DIR,
                      ignore_patterns=['*.bin', '*.msgpack'], max_workers=2)
    print('  ✓ Model downloaded')
else:
    print('  ✓ Model already cached')

size_gb = sum(os.path.getsize(os.path.join(dp, f))
              for dp, _, filenames in os.walk(CACHE_DIR) for f in filenames) / 1024**3
print(f'  Cache size: {size_gb:.1f} GB')
print('  ✓ Ready')

---
## Step 4: Write the Pipeline Script

Creates `run_ltx_video.py` — the complete pipeline with Diffusers + Gradio.

In [ ]:
%%writefile run_ltx_video.py
"""
LTX-Video Generator — Kaggle T4 GPU Edition
Text-to-Video & Image-to-Video using HuggingFace Diffusers + Gradio
"""
import gc, os, sys, json, time, traceback, glob
import subprocess
from pathlib import Path
from datetime import datetime

import numpy as np
import torch
import gradio as gr
from PIL import Image
import imageio

# ── Config ──────────────────────────────────────────────
MODEL_ID = 'Lightricks/LTX-Video'
CACHE_DIR = '/kaggle/working/model_cache'
OUTPUT_DIR = Path('/kaggle/working/generated_videos')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
GALLERY_FILE = OUTPUT_DIR / 'gallery.json'

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,garbage_collection_threshold:0.6'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

print(f'  PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU: {torch.cuda.get_device_name(0)}  |  VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')

# ── Gallery Helpers ────────────────────────────────────
def load_gallery():
    if GALLERY_FILE.exists():
        return json.loads(GALLERY_FILE.read_text())
    return []

def save_gallery_entry(video_path, prompt, mode, params):
    gallery = load_gallery()
    gallery.append({
        'video': str(video_path), 'prompt': prompt,
        'mode': mode, 'params': params,
        'timestamp': datetime.now().isoformat()
    })
    if len(gallery) > 20:
        for old in gallery[:-20]:
            old_path = Path(old['video'])
            if old_path.exists():
                old_path.unlink()
        gallery = gallery[-20:]
    GALLERY_FILE.write_text(json.dumps(gallery, indent=2))
    return gallery

# ── Model Loading ──────────────────────────────────────
pipe = None
pipe_i2v = None

def load_models():
    global pipe, pipe_i2v
    from diffusers import LTXVideoPipeline, LTXImageToVideoPipeline

    print('  Loading LTX-Video pipelines...')

    pipe = LTXVideoPipeline.from_pretrained(
        CACHE_DIR, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True
    )
    pipe.enable_sequential_cpu_offload()
    print('  ✓ T2V pipeline ready')

    pipe_i2v = LTXImageToVideoPipeline.from_pretrained(
        CACHE_DIR, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True
    )
    pipe_i2v.enable_sequential_cpu_offload()
    print('  ✓ I2V pipeline ready')

# ── Generation ─────────────────────────────────────────
def generate_t2v(prompt, negative_prompt, steps, guidance, seed,
                 width, height, num_frames, fps, progress=gr.Progress()):
    global pipe
    if pipe is None:
        load_models()

    progress(0.05, desc='Cleaning up memory...')
    gc.collect()
    torch.cuda.empty_cache()

    generator = torch.Generator(device='cpu').manual_seed(seed) if seed >= 0 else None

    progress(0.1, desc='Generating video...')
    try:
        output = pipe(
            prompt=prompt,
            negative_prompt=negative_prompt or None,
            num_inference_steps=steps,
            guidance_scale=guidance,
            num_frames=num_frames,
            fps=fps,
            width=width,
            height=height,
            generator=generator,
            callback_on_step_end=lambda step, _, __: progress(
                0.1 + 0.7 * (step + 1) / steps,
                desc=f'Step {step + 1}/{steps}'
            )
        )
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        gc.collect()
        raise RuntimeError(
            'VRAM yetersiz! Cozunurlugu dusurun (512x512) veya frame sayisini azaltin (25).'
        )

    progress(0.85, desc='Saving video...')
    frames = output.frames[0]
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    video_path = OUTPUT_DIR / f't2v_{timestamp}.mp4'

    if isinstance(frames[0], Image.Image):
        frames_np = [np.array(f) for f in frames]
    else:
        frames_np = [(f * 255).clip(0, 255).astype(np.uint8) if f.dtype != np.uint8 else f for f in frames]

    imageio.mimsave(str(video_path), frames_np, fps=fps, codec='libx264', quality=8)

    params = {'steps': steps, 'guidance': guidance, 'seed': seed,
              'resolution': f'{width}x{height}', 'frames': num_frames, 'fps': fps}
    save_gallery_entry(video_path, prompt, 'Text-to-Video', params)

    progress(1.0, desc='Done!')
    return str(video_path)

def generate_i2v(image, prompt, negative_prompt, steps, guidance, seed,
                 num_frames, fps, progress=gr.Progress()):
    global pipe_i2v
    if pipe_i2v is None:
        load_models()

    progress(0.05, desc='Cleaning up memory...')
    gc.collect()
    torch.cuda.empty_cache()

    if image is None:
        raise ValueError('Please upload an image for Image-to-Video mode.')

    generator = torch.Generator(device='cpu').manual_seed(seed) if seed >= 0 else None

    progress(0.1, desc='Generating video from image...')
    try:
        output = pipe_i2v(
            image=image,
            prompt=prompt,
            negative_prompt=negative_prompt or None,
            num_inference_steps=steps,
            guidance_scale=guidance,
            num_frames=num_frames,
            fps=fps,
            generator=generator,
            callback_on_step_end=lambda step, _, __: progress(
                0.1 + 0.7 * (step + 1) / steps,
                desc=f'Step {step + 1}/{steps}'
            )
        )
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        gc.collect()
        raise RuntimeError(
            'VRAM yetersiz! Daha kucuk bir gorsel yukleyin veya frame sayisini azaltin.'
        )

    progress(0.85, desc='Saving video...')
    frames = output.frames[0]
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    video_path = OUTPUT_DIR / f'i2v_{timestamp}.mp4'

    if isinstance(frames[0], Image.Image):
        frames_np = [np.array(f) for f in frames]
    else:
        frames_np = [(f * 255).clip(0, 255).astype(np.uint8) if f.dtype != np.uint8 else f for f in frames]

    imageio.mimsave(str(video_path), frames_np, fps=fps, codec='libx264', quality=8)

    params = {'steps': steps, 'guidance': guidance, 'seed': seed,
              'frames': num_frames, 'fps': fps}
    save_gallery_entry(video_path, prompt, 'Image-to-Video', params)

    progress(1.0, desc='Done!')
    return str(video_path)

# ── Audio Mixing ───────────────────────────────────────
def mix_audio_video(video_path, audio_path, volume=1.0):
    if not audio_path or not video_path:
        return video_path

    out_path = str(Path(video_path).with_suffix('.mixed.mp4'))
    cmd = [
        'ffmpeg', '-y', '-i', video_path, '-i', audio_path,
        '-c:v', 'copy', '-c:a', 'aac',
        '-af', f'volume={volume}',
        '-shortest', out_path
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f'  ⚠ Audio mix failed: {result.stderr[:200]}')
        return video_path
    return out_path

# ── Gradio UI ──────────────────────────────────────────
def build_ui():
    css = """
    .gradio-container { max-width: 1100px !important; margin: auto; }
    .generate-btn { background: linear-gradient(135deg, #6a11cb, #2575fc) !important; color: white !important; font-size: 1.2em !important; }
    .generate-btn:hover { opacity: 0.9; }
    footer { display: none !important; }
    """
    with gr.Blocks(css=css, title='LTX-Video Generator', theme=gr.themes.Soft()) as demo:
        gr.Markdown("""
        # 🎬 LTX-Video Generator
        ### Text-to-Video & Image-to-Video | Lightricks LTX-Video + Diffusers | T4 GPU
        """)

        with gr.Tabs():
            with gr.Tab('📝 Text-to-Video'):
                with gr.Row():
                    with gr.Column(scale=2):
                        prompt = gr.Textbox(
                            label='Prompt', lines=3,
                            placeholder='A cinematic shot of a cat walking through a neon-lit cyberpunk city at night...',
                            info='Describe the video you want to generate in detail.'
                        )
                        negative_prompt = gr.Textbox(
                            label='Negative Prompt', lines=2,
                            placeholder='low quality, blurry, distorted, ugly, watermark...',
                            info='What to avoid in the generated video.'
                        )
                        with gr.Row():
                            with gr.Column():
                                steps = gr.Slider(8, 50, value=30, step=1, label='Inference Steps')
                                guidance = gr.Slider(1.0, 10.0, value=3.5, step=0.5, label='Guidance Scale')
                                seed = gr.Number(value=42, label='Seed (-1 = random)', precision=0)
                            with gr.Column():
                                width = gr.Slider(256, 768, value=512, step=64, label='Width')
                                height = gr.Slider(256, 768, value=512, step=64, label='Height')
                                num_frames = gr.Slider(9, 97, value=49, step=8, label='Number of Frames')
                                fps = gr.Slider(8, 30, value=24, step=1, label='FPS')

                        audio_file = gr.Audio(label='Background Audio (optional)', type='filepath')
                        audio_volume = gr.Slider(0.1, 2.0, value=0.5, step=0.1, label='Audio Volume')

                        gen_btn = gr.Button('🎬 Generate Video', variant='primary', elem_classes='generate-btn')

                    with gr.Column(scale=1):
                        video_output = gr.Video(label='Generated Video', height=400)

                gen_btn.click(
                    fn=generate_t2v,
                    inputs=[prompt, negative_prompt, steps, guidance, seed,
                            width, height, num_frames, fps],
                    outputs=video_output
                ).then(
                    fn=mix_audio_video,
                    inputs=[video_output, audio_file, audio_volume],
                    outputs=video_output
                )

            with gr.Tab('🖼️ Image-to-Video'):
                with gr.Row():
                    with gr.Column(scale=2):
                        input_image = gr.Image(label='Input Image', type='pil', height=300)
                        prompt_i2v = gr.Textbox(
                            label='Prompt', lines=2,
                            placeholder='The person smiles and waves at the camera...',
                            info='Describe the motion you want from this image.'
                        )
                        negative_prompt_i2v = gr.Textbox(
                            label='Negative Prompt', lines=1,
                            placeholder='low quality, blurry, distorted...'
                        )
                        with gr.Row():
                            with gr.Column():
                                steps_i2v = gr.Slider(8, 50, value=30, step=1, label='Inference Steps')
                                guidance_i2v = gr.Slider(1.0, 10.0, value=3.5, step=0.5, label='Guidance Scale')
                                seed_i2v = gr.Number(value=42, label='Seed (-1 = random)', precision=0)
                            with gr.Column():
                                num_frames_i2v = gr.Slider(9, 97, value=49, step=8, label='Number of Frames')
                                fps_i2v = gr.Slider(8, 30, value=24, step=1, label='FPS')

                        audio_file_i2v = gr.Audio(label='Background Audio (optional)', type='filepath')
                        audio_volume_i2v = gr.Slider(0.1, 2.0, value=0.5, step=0.1, label='Audio Volume')

                        gen_btn_i2v = gr.Button('🎬 Generate Video', variant='primary', elem_classes='generate-btn')

                    with gr.Column(scale=1):
                        video_output_i2v = gr.Video(label='Generated Video', height=400)

                gen_btn_i2v.click(
                    fn=generate_i2v,
                    inputs=[input_image, prompt_i2v, negative_prompt_i2v, steps_i2v,
                            guidance_i2v, seed_i2v, num_frames_i2v, fps_i2v],
                    outputs=video_output_i2v
                ).then(
                    fn=mix_audio_video,
                    inputs=[video_output_i2v, audio_file_i2v, audio_volume_i2v],
                    outputs=video_output_i2v
                )

            with gr.Tab('🖼️ Gallery'):
                gr.Markdown('### Previously Generated Videos (last 20)')

                gallery_list = gr.Dataframe(
                    headers=['Timestamp', 'Mode', 'Prompt', 'Params', 'Video'],
                    datatype=['str', 'str', 'str', 'str', 'str'],
                    label='Generation History',
                    interactive=False
                )

                def refresh_gallery():
                    gallery = load_gallery()
                    rows = []
                    for g in reversed(gallery):
                        rows.append([
                            g['timestamp'][:19],
                            g['mode'],
                            g['prompt'][:80] + ('...' if len(g['prompt']) > 80 else ''),
                            json.dumps(g['params']),
                            g['video']
                        ])
                    return rows

                refresh_btn = gr.Button('🔄 Refresh Gallery')
                refresh_btn.click(fn=refresh_gallery, outputs=gallery_list)

        gr.Markdown("---\n<p style='text-align: center; color: #888;'>LTX-Video Generator — Powered by HuggingFace Diffusers</p>")

    return demo

# ── Main ───────────────────────────────────────────────
if __name__ == '__main__':
    print('=' * 50)
    print('  LTX-Video Generator — Starting...')
    print('=' * 50)

    demo = build_ui()
    demo.queue(max_size=5).launch(share=True, debug=False)

---
## Step 5: Launch!

Runs the generation script. Look for the **public Gradio URL** in the output.

In [ ]:
!python -u run_ltx_video.py 2>&1

---

<div align='center'>

### 🎉 Enjoyed this notebook?

<!-- TODO: Add your own branding, logo, and social links here -->

<p style='color: #888;'>Created with ❤️ using HuggingFace Diffusers + Gradio</p>

</div>